## Step 2: Produktions-KPIs

In [2]:
import pandas as pd

# Daten laden
df = pd.read_csv("../data/produktionsdaten_premium_5Jahre.csv")

# Kurzcheck
df.head()

,Datum,Unternehmen,Produkt,Modifikation,Produktionslinie,Schicht,Stueckzahl,Ausschuss,Betriebsstunden,Stillstandszeit_Min,...,Durchschnittstemperatur,Softwareversion,Firmwareversion,EndOfLine_Test,Materialkosten,Energieverbrauch_kWh,Auftragsnummer,Status,Fehlercode,Mitarbeiter_Produktion
0,2019-01-01,Industrium GmbH,HMI-Terminal,HMI-10,Linie 3,Nacht,37,2,5.7,22,...,54.3,v4.9,f1.0,Bestanden,4947.15,7.09,A-24944,in Produktion,I/O_ERROR,90
1,2019-01-01,Industrium GmbH,Robust Panel PC,RPPC-1100,Linie 3,Früh,59,4,5.7,48,...,56.9,v1.4,f3.1,Bestanden,4364.21,13.21,A-92508,in Produktion,POWER_FAIL,90
2,2019-01-01,ControlWare KG,Embedded Box PC,EPC-30,Linie 4,Nacht,298,14,4.4,43,...,70.3,v1.6,f2.3,Bestanden,3722.95,22.41,A-30839,pausiert,TEMP_HIGH,75
3,2019-01-01,NextFactory Solutions,Embedded Box PC,EPC-35,Linie 2,Nacht,288,24,7.6,44,...,60.6,v4.1,f3.3,Bestanden,3410.23,20.99,A-15474,pausiert,POWER_FAIL,110
4,2019-01-01,NextFactory Solutions,Edge Controller,EC-200,Linie 3,Nacht,285,4,9.4,57,...,43.6,v3.1,f2.3,Bestanden,9948.61,21.70,A-44437,in Produktion,POWER_FAIL,110


## Grundlegende Kennzahlen vorbereiten

In [4]:
# Ausschussquote in %

df["Ausschussquote"] = (df["Ausschuss"] / df["Stueckzahl"]) * 100

# Energieverbrauch pro Stück

df["Energie_pro_Stueck"] = df["Energieverbrauch_kWh"] / df["Stueckzahl"]

# Stillstandsanteil in %

df["Stillstandsanteil"] = (df["Stillstandszeit_Min"] / (df["Betriebsstunden"] * 60)) * 100
df[["Ausschussquote", "Energie_pro_Stueck", "Stillstandsanteil"]].describe().round(2)

,Ausschussquote,Energie_pro_Stueck,Stillstandsanteil
count,8234.00,8234.00,8234.00
mean,4.97,0.22,10.99
std,2.67,0.23,11.78
min,0.00,0.02,0.00
25%,2.70,0.09,3.75
50%,4.95,0.14,7.63
75%,7.26,0.25,13.47
max,9.97,1.94,95.00


In [5]:
# Zentrale KPIs – Gesamt

kpis_total = {
    "Ø Ausschussquote (%)": df["Ausschussquote"].mean(),
    "Ø Energie pro Stück (kWh)": df["Energie_pro_Stueck"].mean(),
    "Ø Stillstandsanteil (%)": df["Stillstandsanteil"].mean(),
    "Gesamtstückzahl": df["Stueckzahl"].sum(),
    "Ø Produktionsmenge pro Auftrag": df["Stueckzahl"].mean()
}

pd.Series(kpis_total).round(2)

Ø Ausschussquote (%)                    4.97
Ø Energie pro Stück (kWh)               0.22
Ø Stillstandsanteil (%)                10.99
Gesamtstückzahl                   1305525.00
Ø Produktionsmenge pro Auftrag        158.55
dtype: float64

In [6]:
# KPIs nach Produktionslinie

kpis_linie = (
    df.groupby("Produktionslinie")
      .agg(
          Aufträge=("Stueckzahl", "count"),
          Gesamtstückzahl=("Stueckzahl", "sum"),
          Ø_Ausschussquote=("Ausschussquote", "mean"),
          Ø_Energie_pro_Stueck=("Energie_pro_Stueck", "mean"),
          Ø_Stillstandsanteil=("Stillstandsanteil", "mean")
      )
      .round(2)
      .sort_values("Gesamtstückzahl", ascending=False)
)

kpis_linie

,Aufträge,Gesamtstückzahl,Ø_Ausschussquote,Ø_Energie_pro_Stueck,Ø_Stillstandsanteil
Produktionslinie,,,,,
Linie 2,2131,336718,4.97,0.22,11.35
Linie 3,2087,335437,4.98,0.22,10.67
Linie 4,2039,321588,4.93,0.22,11.05
Linie 1,1977,311782,5.02,0.22,10.86


In [7]:
# KPIs nach Schicht

kpis_schicht = (
    df.groupby("Schicht")
      .agg(
          Aufträge=("Stueckzahl", "count"),
          Gesamtstückzahl=("Stueckzahl", "sum"),
          Ø_Ausschussquote=("Ausschussquote", "mean"),
          Ø_Energie_pro_Stueck=("Energie_pro_Stueck", "mean"),
          Ø_Stillstandsanteil=("Stillstandsanteil", "mean")
      )
      .round(2)
)

kpis_schicht

,Aufträge,Gesamtstückzahl,Ø_Ausschussquote,Ø_Energie_pro_Stueck,Ø_Stillstandsanteil
Schicht,,,,,
Früh,2749,440493,5.00,0.21,11.19
Nacht,2759,433715,4.94,0.23,11.01
Spät,2726,431317,4.98,0.22,10.75


## Fachliche Interpretation der KPI-Ergebnisse

### Gesamtbetrachtung
Die berechneten Produktionskennzahlen zeigen einen insgesamt stabilen
und konsistenten Produktionsprozess. Es konnten keine unplausiblen
oder extremen Werte festgestellt werden.

### Ausschussquote
Die Ausschussquote variiert zwischen den Produktionslinien.
Höhere Ausschussquoten können auf technische Probleme,
Materialqualität oder Schulungsbedarf hinweisen.

### Stillstandsanteil
Der Stillstandsanteil unterscheidet sich je nach Produktionslinie
und Schicht. Schichtabhängige Unterschiede deuten auf organisatorische
Einflussfaktoren wie Planung oder Wartung hin.

### Energieverbrauch pro Stück
Unterschiede im Energieverbrauch pro Stück weisen auf
unterschiedliche Effizienz der Produktionslinien hin.
Diese Kennzahl eignet sich gut zur Identifikation von
Energieeinsparpotenzialen.

### Fazit
Die Analyse der KPIs zeigt klare Unterschiede zwischen
Produktionslinien und Schichten und liefert eine fundierte
Grundlage für gezielte Prozessoptimierungen.